In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
metrics = test_results['global_metrics']['mAP_metrics']

print("\n" + "="*60)
print("PERFORMANCES GLOBALES")
print("="*60)
print(f"mAP: {metrics['map']:.4f}")
print(f"mAP@50: {metrics['map_50']:.4f}")
print(f"mAP@75: {metrics['map_75']:.4f}")
print(f"mAR (mean Average Recall): {metrics['mar_100']:.4f}")

In [ ]:
epochs = [r['epoch'] for r in train_results]
train_losses = [r['train']['loss_total'] for r in train_results]
val_losses = [r['val']['loss_total'] for r in train_results]

train_cls = [r['train']['loss_classifier'] for r in train_results]
train_box = [r['train']['loss_box_reg'] for r in train_results]
train_obj = [r['train']['loss_objectness'] for r in train_results]
train_rpn = [r['train']['loss_rpn_box_reg'] for r in train_results]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss totale
axes[0].plot(epochs, train_losses, label='Train', marker='o')
axes[0].plot(epochs, val_losses, label='Validation', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Totale')
axes[0].legend()
axes[0].grid(True)

# Losses individuelles (train)
axes[1].plot(epochs, train_cls, label='Classifier', marker='o')
axes[1].plot(epochs, train_box, label='Box Regression', marker='s')
axes[1].plot(epochs, train_obj, label='Objectness', marker='^')
axes[1].plot(epochs, train_rpn, label='RPN Box Reg', marker='d')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Losses Individuelles (Train)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('../outputs/losses_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
map_per_class = metrics['map_per_class']
mar_per_class = metrics['mar_100_per_class']
num_classes = len(map_per_class)

# Tableau recapitulatif
print("\n" + "="*60)
print("PERFORMANCES PAR CLASSE")
print("="*60)
print(f"{'Classe':<10} {'AP':>8} {'AR':>8}")
print("-"*60)
for i in range(num_classes):
    print(f"{i:<10} {map_per_class[i]:>8.4f} {mar_per_class[i]:>8.4f}")

# Bar plot AP par classe
fig, ax = plt.subplots(figsize=(10, 6))
classes = [f"Classe {i}" for i in range(num_classes)]
x_pos = np.arange(num_classes)
ax.bar(x_pos, map_per_class, alpha=0.8)
ax.set_xlabel('Classe')
ax.set_ylabel('Average Precision (AP)')
ax.set_title('AP par Classe')
ax.set_xticks(x_pos)
ax.set_xticklabels(classes, rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/ap_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    
    return inter / union if union > 0 else 0

# Initialisation de la matrice (classes + Missed)
confusion_matrix = np.zeros((num_classes + 1, num_classes + 1), dtype=int)
# Derniere ligne pour FP, derniere colonne pour Missed

for result in test_results['detailed_results']:
    gt_boxes = result['ground_truth']['boxes']
    gt_labels = result['ground_truth']['labels']
    pred_boxes = result['predictions']['boxes']
    pred_labels = result['predictions']['labels']
    pred_scores = result['predictions']['scores']
    
    matched_preds = set()
    
    # Pour chaque GT, trouver la meilleure prediction
    for gt_idx, (gt_box, gt_label) in enumerate(zip(gt_boxes, gt_labels)):
        best_iou = 0
        best_pred_idx = -1
        
        for pred_idx, pred_box in enumerate(pred_boxes):
            if pred_idx in matched_preds:
                continue
            iou = compute_iou(gt_box, pred_box)
            if iou > best_iou:
                best_iou = iou
                best_pred_idx = pred_idx
        
        if best_iou > 0.5 and best_pred_idx != -1:
            pred_label = pred_labels[best_pred_idx]
            confusion_matrix[gt_label][pred_label] += 1
            matched_preds.add(best_pred_idx)
        else:
            # Non detecte -> colonne "Missed"
            confusion_matrix[gt_label][num_classes] += 1
    
    # Predictions non appariees = FP (derniere ligne)
    for pred_idx in range(len(pred_boxes)):
        if pred_idx not in matched_preds:
            pred_label = pred_labels[pred_idx]
            confusion_matrix[num_classes][pred_label] += 1

# Conversion en pourcentages par ligne
confusion_matrix_pct = np.zeros_like(confusion_matrix, dtype=float)
for i in range(confusion_matrix.shape[0]):
    row_sum = confusion_matrix[i].sum()
    if row_sum > 0:
        confusion_matrix_pct[i] = (confusion_matrix[i] / row_sum) * 100

# Affichage
labels = [f"C{i}" for i in range(num_classes)] + ["Missed"]
labels_y = [f"C{i}" for i in range(num_classes)] + ["FP"]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(confusion_matrix_pct, annot=True, fmt='.1f', cmap='Blues', 
            xticklabels=labels, yticklabels=labels_y, 
            cbar_kws={'label': 'Pourcentage (%)'}, ax=ax)
ax.set_xlabel('Predictions')
ax.set_ylabel('Ground Truth')
ax.set_title('Matrice de Confusion (en %)')
plt.tight_layout()
plt.savefig('../outputs/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nAnalyse terminee. Graphiques sauvegardes dans ../outputs/")